# Notebook: 01-06-MLFlow -basics: MLFlow Integration and Performance Analysis Notebook
___
## Overview
This notebook demonstrates how to use MLFlow to track and analyze the performance of different nodes in a conversational AI system. It focuses on gathering statistics about execution times and interaction patterns across multiple experiments.


## Section 1: Setup and Configuration


In [1]:
import os
import sys
import json
import mlflow
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
import pandas as pd
import mlflow
import time

In [2]:
from pprint import pprint

In [3]:
# Get the absolute path to the backend directory
current_dir = os.getcwd()  # Get current working directory
backend_path = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(backend_path)

# Load environment variables from .env file
load_dotenv()

True

In [4]:
try:
    mlflow_arn = os.getenv('MLFLOW_URI')
    if mlflow_arn:
        print(f'\nMLFlow tracking URI: {mlflow_arn}')
        mlflow.set_tracking_uri(mlflow_arn)
        print('Successfully connected to MLFlow tracking server')
except Exception as e:
    print('Error connecting to MLFlow tracking server')
    raise e


MLFlow tracking URI: arn:aws:sagemaker:us-west-2:577201992296:mlflow-tracking-server/bedrock-chatbot-mlflow
Successfully connected to MLFlow tracking server


In [5]:
# Calculate the timestamp for 'n' hours ago in milliseconds
hours_back = 24
n_hours_ago = int((time.time() - hours_back * 60 * 60) * 1000)

# Use mlflow.search_experiments with the numeric timestamp value
experiments = mlflow.search_experiments(
    filter_string=f"last_update_time > {n_hours_ago}"  # Remove the parentheses
)

# Display the resulting experiments
experiments_name = []
for experiment in experiments:
    print(f"Experiment ID: {experiment.experiment_id}, Name: {experiment.name}")
    experiments_name.append(experiment.name)

Experiment ID: 321, Name: bf573390-4914-47f8-87d3-22f3f1f1f7a9
Experiment ID: 318, Name: 6c212ea0-c296-4812-b84b-b1f7e05bfbad
Experiment ID: 317, Name: 912760af-64ae-4126-b96a-375bc2809594


In [6]:
experiment_id = experiments[0].experiment_id
traces = mlflow.search_traces(
    experiment_ids=[experiment_id],
    filter_string="status = 'COMPLETED'"  # You can filter by supported attributes
)

In [7]:
# Retrieve the experiment by name to get the experiment ID
experiment_name = experiments[0].name
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment:
    # Use mlflow.search_traces to find traces within the specified experiment
    traces_df = mlflow.search_traces(
        experiment_ids=[experiment.experiment_id]
    )
    # Display the resulting DataFrame
    print(traces_df)
else:
    print(f"Experiment with name '(experiment_name)' not found.")

                         request_id  \
0  87ff9982783f4016a3e349c67012bff1   
1  f87605238c89489f87b43117b1ef346c   
2  de3aa34094ea4c9cb3a9010712ea97ca   
3  3e56e92c1c38409cbf9b29c842956914   

                                               trace   timestamp_ms  \
0  Trace(request_id=87ff9982783f4016a3e349c67012b...  1742410534538   
1  Trace(request_id=f87605238c89489f87b43117b1ef3...  1742410510379   
2  Trace(request_id=de3aa34094ea4c9cb3a9010712ea9...  1742410412200   
3  Trace(request_id=3e56e92c1c38409cbf9b29c842956...  1742410385481   

           status  execution_time_ms  \
0  TraceStatus.OK              16680   
1  TraceStatus.OK              15796   
2  TraceStatus.OK              15389   
3  TraceStatus.OK               2033   

                                             request  \
0  {'messages': [{'role': 'user', 'content': [{'t...   
1  {'messages': [{'role': 'user', 'content': [{'t...   
2  {'messages': [{'role': 'user', 'content': [{'t...   
3  {'messages': [{'role

In [8]:
traces_df

,request_id,trace,timestamp_ms,status,execution_time_ms,request,response,request_metadata,spans,tags
0,87ff9982783f4016a3e349c67012bff1,Trace(request_id=87ff9982783f4016a3e349c67012b...,1742410534538,TraceStatus.OK,16680,"{'messages': [{'role': 'user', 'content': [{'t...","{'messages': [{'role': 'user', 'content': [{'t...","{'mlflow.traceInputs': '{""messages"": [{""role"":...","[{'name': 'LangGraph', 'context': {'span_id': ...",{'mlflow.source.name': '/opt/conda/lib/python3...
1,f87605238c89489f87b43117b1ef346c,Trace(request_id=f87605238c89489f87b43117b1ef3...,1742410510379,TraceStatus.OK,15796,"{'messages': [{'role': 'user', 'content': [{'t...","{'messages': [{'role': 'user', 'content': [{'t...","{'mlflow.trace_schema.version': '2', 'mlflow.t...","[{'name': 'LangGraph', 'context': {'span_id': ...",{'mlflow.source.name': '/opt/conda/lib/python3...
2,de3aa34094ea4c9cb3a9010712ea97ca,Trace(request_id=de3aa34094ea4c9cb3a9010712ea9...,1742410412200,TraceStatus.OK,15389,"{'messages': [{'role': 'user', 'content': [{'t...","{'messages': [{'role': 'user', 'content': [{'t...","{'mlflow.trace_schema.version': '2', 'mlflow.t...","[{'name': 'LangGraph', 'context': {'span_id': ...",{'mlflow.source.name': '/opt/conda/lib/python3...
3,3e56e92c1c38409cbf9b29c842956914,Trace(request_id=3e56e92c1c38409cbf9b29c842956...,1742410385481,TraceStatus.OK,2033,"{'messages': [{'role': 'user', 'content': [{'t...","{'messages': [{'role': 'user', 'content': [{'t...","{'mlflow.traceInputs': '{""messages"": [{""role"":...","[{'name': 'LangGraph', 'context': {'span_id': ...",{'mlflow.source.name': '/opt/conda/lib/python3...


In [9]:
# Convert execution time from milliseconds to seconds and calculate statistics
stats = {
    'min_time': traces_df['execution_time_ms'].min() / 1000,  # Convert to seconds
    'max_time': traces_df['execution_time_ms'].max() / 1000,
    'avg_time': traces_df['execution_time_ms'].mean() / 1000,
    'median_time': traces_df['execution_time_ms'].median() / 1000,
    'std_time': traces_df['execution_time_ms'].std() / 1000
}

# Print the statistics
print("\n ---- Execution Time Statistics (in seconds): ---- \n")
print(f"Minimum: {stats['min_time']:.2f} sec")
print(f"Maximum: {stats['max_time']:.2f} sec")
print(f"Average: {stats['avg_time']:.2f} sec")
print(f"Median: {stats['median_time']:.2f} sec")
print(f"Standard Deviation: {stats['std_time']:.2f} sec")


 ---- Execution Time Statistics (in seconds): ---- 

Minimum: 2.03 sec
Maximum: 16.68 sec
Average: 12.47 sec
Median: 15.59 sec
Standard Deviation: 6.98 sec


In [10]:
traces_df.iloc[0]

request_id                            87ff9982783f4016a3e349c67012bff1
trace                Trace(request_id=87ff9982783f4016a3e349c67012b...
timestamp_ms                                             1742410534538
status                                                  TraceStatus.OK
execution_time_ms                                                16680
request              {'messages': [{'role': 'user', 'content': [{'t...
response             {'messages': [{'role': 'user', 'content': [{'t...
request_metadata     {'mlflow.traceInputs': '{"messages": [{"role":...
spans                [{'name': 'LangGraph', 'context': {'span_id': ...
tags                 {'mlflow.source.name': '/opt/conda/lib/python3...
Name: 0, dtype: object

# Latency Metrics Methods
___

In [11]:
def get_execution_stats(experiments):
    all_execution_times = []
    
    for experiment in experiments:
        experiment_name = experiment.name
        exp = mlflow.get_experiment_by_name(experiment_name)
        
        if exp:
            traces_df = mlflow.search_traces(
                experiment_ids=[exp.experiment_id]
            )
            
            # Collect execution times from this experiment
            all_execution_times.extend(traces_df['execution_time_ms'].tolist())
    
    # Convert to pandas Series for easy calculation
    execution_series = pd.Series(all_execution_times)
    
    stats = {
        'min_time': execution_series.min() / 1000,
        'max_time': execution_series.max() / 1000,
        'avg_time': execution_series.mean() / 1000,
        'median_time': execution_series.median() / 1000,
        'std_time': execution_series.std() / 1000
    }
    
    print("\n ---- Execution Time Statistics Across All Experiments (in seconds): ---- \n")
    print(f"Minimum: {stats['min_time']:.2f} sec")
    print(f"Maximum: {stats['max_time']:.2f} sec")
    print(f"Average: {stats['avg_time']:.2f} sec")
    print(f"Median: {stats['median_time']:.2f} sec")
    print(f"Standard Deviation: {stats['std_time']:.2f} sec")
    print(f"Total Sessions: {len(experiments)}")
    
    return stats


def extract_trace_info(trace_df_row):
    """
    Extract trace information including langgraph_node, start_time, end_time, and duration
    Args:
        trace_df_row: DataFrame row containing trace information
    Returns:
        List of dictionaries containing trace information
    """
    trace_info = []
    
    for idx, span in enumerate(trace_df_row['spans']):
        metadata = span.get('attributes', {}).get('metadata')
        if metadata:
            try:
                langgraph_node_content = json.loads(metadata)
                langgraph_node = langgraph_node_content.get('langgraph_node')
                spanType = span.get('attributes').get('mlflow.spanType')
                trace_node_name = span.get('name')
                
                if (spanType.strip('"') == "CHAIN" and 
                    langgraph_node != '__start__' and 
                    trace_node_name == langgraph_node):
                    
                    start_time = span.get('start_time')
                    end_time = span.get('end_time')
                    duration = (end_time - start_time) / 1e9
                    
                    trace_info.append({
                        'langgraph_node': langgraph_node,
                        'start_time': start_time,
                        'end_time': end_time,
                        'duration': duration
                    })
            except json.JSONDecodeError:
                continue
                
    return trace_info


def aggregate_trace_stats(traces_df):
    all_trace_info = []
    for trace_idx in range(len(traces_df)):
        trace_info = extract_trace_info(traces_df.iloc[trace_idx])
        all_trace_info.extend(trace_info)
    
    # Group by langgraph_node and calculate stats
    node_stats = {}
    for info in all_trace_info:
        node = info['langgraph_node']
        if node not in node_stats:
            node_stats[node] = {'durations': []}
        node_stats[node]['durations'].append(info['duration'])
    
    # Calculate statistics
    for node in node_stats:
        durations = node_stats[node]['durations']
        node_stats[node].update({
            'count': len(durations),
            'min_duration': min(durations),
            'max_duration': max(durations),
            'avg_duration': sum(durations) / len(durations),
            'total_duration': sum(durations)
        })
        del node_stats[node]['durations']  # Remove raw durations after calculating stats
        
    return node_stats

def aggregate_experiment_stats(experiments):
    """
    Aggregates statistics across multiple experiments
    Args:
        experiments: List of MLflow experiment objects
    Returns:
        Two dictionaries: per_experiment_stats and total_stats
    """
    per_experiment_stats = {}
    all_node_stats = {}
    
    for experiment in experiments:
        # Get traces for this experiment
        traces_df = mlflow.search_traces(
            experiment_ids=[experiment.experiment_id]
        )
        
        print(f"Processing experiment: {experiment.name}")
        print(f"Found {len(traces_df)} traces")
        
        if len(traces_df) > 0:
            # Use aggregate_trace_stats to compute statistics for this experiment
            experiment_stats = aggregate_trace_stats(traces_df)
            per_experiment_stats[experiment.name] = experiment_stats
            
            # Accumulate stats for total aggregation
            for node, stats in experiment_stats.items():
                if node not in all_node_stats:
                    all_node_stats[node] = {'durations': []}
                all_node_stats[node]['durations'].extend([stats['avg_duration']] * stats['count'])
    
    # Calculate total statistics
    total_stats = {}
    for node in all_node_stats:
        durations = all_node_stats[node]['durations']
        total_stats[node] = {
            'count': len(durations),
            'min_duration': min(durations),
            'max_duration': max(durations),
            'avg_duration': sum(durations) / len(durations),
            'total_duration': sum(durations)
        }
    
    return per_experiment_stats, total_stats




# Compute Latency Metrics
___

In [12]:
# Usage:
trace_info = extract_trace_info(traces_df.iloc[0])
for info in trace_info:
    print(f"Node: {info['langgraph_node']}")
    print(f"Start time: {info['start_time']}")
    print(f"End time: {info['end_time']}")
    print(f"Duration: {info['duration']:.2f} seconds\n")

Node: order_confirmation
Start time: 1742410534609856426
End time: 1742410536570538326
Duration: 1.96 seconds

Node: resolution
Start time: 1742410536571133846
End time: 1742410551218504220
Duration: 14.65 seconds



In [13]:
# Usage:
stats = aggregate_trace_stats(traces_df)
for node, node_stats in stats.items():
    print(f"\nNode: {node}")
    print(f"Count: {node_stats['count']}")
    print(f"Min duration: {node_stats['min_duration']:.2f} seconds")
    print(f"Max duration: {node_stats['max_duration']:.2f} seconds")
    print(f"Average duration: {node_stats['avg_duration']:.2f} seconds")
    print(f"Total duration: {node_stats['total_duration']:.2f} seconds")


Node: order_confirmation
Count: 3
Min duration: 1.96 seconds
Max duration: 5.30 seconds
Average duration: 3.66 seconds
Total duration: 10.97 seconds

Node: resolution
Count: 2
Min duration: 6.38 seconds
Max duration: 14.65 seconds
Average duration: 10.51 seconds
Total duration: 21.02 seconds

Node: entry_intent
Count: 3
Min duration: 1.96 seconds
Max duration: 10.01 seconds
Average duration: 5.87 seconds
Total duration: 17.60 seconds


In [14]:
# Get stats for the experiment
per_experiment_stats, total_stats = aggregate_experiment_stats(experiments)

# Print per-experiment statistics
print("Per Experiment Statistics:")
for experiment_name, stats in per_experiment_stats.items():
    print(f"\nExperiment: {experiment_name}")
    for node, node_stats in stats.items():
        print(f"\n  Node: {node}")
        print(f"  Count: {node_stats['count']}")
        print(f"  Min duration: {node_stats['min_duration']:.2f} seconds")
        print(f"  Max duration: {node_stats['max_duration']:.2f} seconds")
        print(f"  Average duration: {node_stats['avg_duration']:.2f} seconds")
        print(f"  Total duration: {node_stats['total_duration']:.2f} seconds")


Processing experiment: bf573390-4914-47f8-87d3-22f3f1f1f7a9
Found 4 traces
Processing experiment: 6c212ea0-c296-4812-b84b-b1f7e05bfbad
Found 5 traces
Processing experiment: 912760af-64ae-4126-b96a-375bc2809594
Found 5 traces
Per Experiment Statistics:

Experiment: bf573390-4914-47f8-87d3-22f3f1f1f7a9

  Node: order_confirmation
  Count: 3
  Min duration: 1.96 seconds
  Max duration: 5.30 seconds
  Average duration: 3.66 seconds
  Total duration: 10.97 seconds

  Node: resolution
  Count: 2
  Min duration: 6.38 seconds
  Max duration: 14.65 seconds
  Average duration: 10.51 seconds
  Total duration: 21.02 seconds

  Node: entry_intent
  Count: 3
  Min duration: 1.96 seconds
  Max duration: 10.01 seconds
  Average duration: 5.87 seconds
  Total duration: 17.60 seconds

Experiment: 6c212ea0-c296-4812-b84b-b1f7e05bfbad

  Node: order_confirmation
  Count: 3
  Min duration: 11.58 seconds
  Max duration: 17.47 seconds
  Average duration: 14.85 seconds
  Total duration: 44.55 seconds

  Node:

In [15]:
# Print total statistics across all experiments
print("\nTotal Statistics Across All Experiments:")
for node, node_stats in total_stats.items():
    print(f"\nNode: {node}")
    print(f"Total Count: {node_stats['count']}")
    print(f"Min duration: {node_stats['min_duration']:.2f} seconds")
    print(f"Max duration: {node_stats['max_duration']:.2f} seconds")
    print(f"Average duration: {node_stats['avg_duration']:.2f} seconds")
    print(f"Total duration: {node_stats['total_duration']:.2f} seconds")


Total Statistics Across All Experiments:

Node: order_confirmation
Total Count: 10
Min duration: 3.66 seconds
Max duration: 14.85 seconds
Average duration: 9.66 seconds
Total duration: 96.56 seconds

Node: resolution
Total Count: 7
Min duration: 10.51 seconds
Max duration: 17.12 seconds
Average duration: 14.84 seconds
Total duration: 103.86 seconds

Node: entry_intent
Total Count: 9
Min duration: 4.11 seconds
Max duration: 10.65 seconds
Average duration: 7.60 seconds
Total duration: 68.39 seconds


# Tokens Usage Methods
___

In [16]:
def analyze_trace(trace_df_row):
    chat_model_calls = 0
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0

    for idx, span in enumerate(trace_df_row['spans']):
        spanType = span.get('attributes', {}).get('mlflow.spanType')

        if spanType and spanType.strip('"') == "CHAT_MODEL":
            chat_model_calls += 1

            # Get token usage from mlflow.spanOutputs
            json_string = span['attributes']['mlflow.spanOutputs']
            data = json.loads(json_string)

            # Extract token information from first element which contains ResponseMetadata
            usage = data[0].get('usage', {})
            input_tokens = usage.get('inputTokens', 0)
            output_tokens = usage.get('outputTokens', 0)
            tokens = usage.get('totalTokens', 0)

            total_input_tokens += input_tokens
            total_output_tokens += output_tokens
            total_tokens += tokens

    return {
        'num_llm_calls': chat_model_calls,
        'total_input_tokens': total_input_tokens,
        'total_output_tokens': total_output_tokens,
        'total_tokens': total_tokens
    }



def analyze_all_traces(traces_df):
    total_stats = {
        'num_llm_calls': 0,
        'total_input_tokens': 0,
        'total_output_tokens': 0,
        'total_tokens': 0
    }

    # Iterate through all traces in the DataFrame
    for i in range(traces_df.shape[0]):
        trace_results = analyze_trace(traces_df.iloc[i])

        # Accumulate the statistics
        total_stats['num_llm_calls'] += trace_results['num_llm_calls']
        total_stats['total_input_tokens'] += trace_results['total_input_tokens']
        total_stats['total_output_tokens'] += trace_results['total_output_tokens']
        total_stats['total_tokens'] += trace_results['total_tokens']

    return total_stats


def analyze_all_experiments(experiments):
    total_stats = {
        'num_llm_calls': 0,
        'total_input_tokens': 0,
        'total_output_tokens': 0,
        'total_tokens': 0
    }

    for experiment in experiments:
        # Get experiment traces
        experiment_name = experiment.name
        exp = mlflow.get_experiment_by_name(experiment_name)

        if exp:
            traces_df = mlflow.search_traces(
                experiment_ids=[exp.experiment_id]
            )

            # Analyze traces for this experiment
            experiment_results = analyze_all_traces(traces_df)

            # Accumulate the statistics
            total_stats['num_llm_calls'] += experiment_results['num_llm_calls']
            total_stats['total_input_tokens'] += experiment_results['total_input_tokens']
            total_stats['total_output_tokens'] += experiment_results['total_output_tokens']
            total_stats['total_tokens'] += experiment_results['total_tokens']

    return total_stats

# Compute Tokens Usage
____

In [17]:
# Usage:
results = analyze_trace(traces_df.iloc[0])
print(f"Number of LLM calls: {results['num_llm_calls']}")
print(f"Total input tokens: {results['total_input_tokens']}")
print(f"Total output tokens: {results['total_output_tokens']}")
print(f"Total tokens: {results['total_tokens']}")

Number of LLM calls: 2
Total input tokens: 5445
Total output tokens: 138
Total tokens: 5583


In [18]:
# Usage:
total_results = analyze_all_traces(traces_df)
print(f"Total number of LLM calls across all traces: {total_results['num_llm_calls']}")
print(f"Total input tokens across all traces: {total_results['total_input_tokens']}")
print(f"Total output tokens across all traces: {total_results['total_output_tokens']}")
print(f"Total tokens across all traces: {total_results['total_tokens']}")

Total number of LLM calls across all traces: 14
Total input tokens across all traces: 24766
Total output tokens across all traces: 1312
Total tokens across all traces: 26078


In [19]:
# Usage:
overall_results = analyze_all_experiments(experiments)
print(f"Total LLM calls across all experiments: {overall_results['num_llm_calls']}")
print(f"Total input tokens across all experiments: {overall_results['total_input_tokens']}")
print(f"Total output tokens across all experiments: {overall_results['total_output_tokens']}")
print(f"Total tokens across all experiments: {overall_results['total_tokens']}")

Total LLM calls across all experiments: 42
Total input tokens across all experiments: 72434
Total output tokens across all experiments: 3675
Total tokens across all experiments: 76109
